# Seismic Refraction: Two-Layer Interpretation

**SEES:4800 Environmental Geophysics**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geohang/environmental-geophysics/blob/main/docs/notebooks/seismic_refraction.ipynb)

Pick first-break travel times over a two-layer earth, fit the direct and head-wave
slopes, and recover layer velocities and depth.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## Synthetic first breaks

Geophones every 3 m. True model: V1 = 800 m/s, V2 = 2500 m/s, h = 8 m.


In [ ]:
x = np.arange(3, 96, 3.0)          # offsets (m)
V1, V2, h = 800., 2500., 8.
ti = 2*h*np.sqrt(V2**2 - V1**2)/(V1*V2)
t_direct = x/V1
t_head   = x/V2 + ti
t = np.minimum(t_direct, t_head) * 1000  # ms
t += np.random.default_rng(1).normal(0, 0.4, t.size)  # picking noise


## Fit two straight-line segments

Split the picks at the crossover distance, fit each segment, read the slopes.


In [ ]:
xc = 2*h*np.sqrt((V2+V1)/(V2-V1))
near = x < xc
s1 = np.polyfit(x[near], t[near], 1)   # 1000/V1
s2 = np.polyfit(x[~near], t[~near], 1) # 1000/V2, intercept ti
V1_est = 1000/s1[0]; V2_est = 1000/s2[0]; ti_est = s2[1]/1000
h_est = ti_est/2 * V1_est*V2_est/np.sqrt(V2_est**2 - V1_est**2)
print(f'V1={V1_est:.0f}  V2={V2_est:.0f} m/s  depth={h_est:.1f} m (true 8 m)')


In [ ]:
plt.figure(figsize=(7,4))
plt.plot(x, t, 'ko', ms=4, label='picks')
plt.plot(x, np.polyval(s1,x), label='direct fit')
plt.plot(x, np.polyval(s2,x), label='head-wave fit')
plt.axvline(xc, color='r', ls='--', label='crossover')
plt.xlabel('offset (m)'); plt.ylabel('time (ms)'); plt.legend(); plt.grid(alpha=0.3); plt.show()


## Your turn

1. Increase the picking noise. How stable is the depth estimate?
2. Make V2 closer to V1. Why does the crossover move out?
3. Add a third layer and a second head-wave segment.
